In [6]:
%pip -q install python-dotenv pandas assemblyai

Note: you may need to restart the kernel to use updated packages.


In [7]:
import os
import json
import pandas as pd
import assemblyai as aai

from pathlib import Path

In [8]:
from pathlib import Path

BASE_DIR = Path.cwd()
PROCESSED_DIR = BASE_DIR / "dataset" / "processed"

print("Current directory:", BASE_DIR)
print("Processed directory:", PROCESSED_DIR)
print("Exists:", PROCESSED_DIR.exists())

Current directory: /Users/manish/Documents/KU/SEM 5/DL/Mini Project
Processed directory: /Users/manish/Documents/KU/SEM 5/DL/Mini Project/dataset/processed
Exists: True


In [9]:
audio_files = [
    file for file in PROCESSED_DIR.rglob("*")
    if file.is_file()
]

print(f"Total files: {len(audio_files)}")

for file in audio_files[:20]:
    print(file)

Total files: 100
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/dataset/processed/test/audio_018.wav
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/dataset/processed/test/audio_019.wav
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/dataset/processed/test/audio_009.wav
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/dataset/processed/test/audio_008.wav
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/dataset/processed/test/audio_011.wav
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/dataset/processed/test/audio_005.wav
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/dataset/processed/test/audio_004.wav
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/dataset/processed/test/audio_010.wav
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/dataset/processed/test/audio_006.wav
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/dataset/processed/test/audio_012.wav
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/dataset/processed/test/audio_013.wav
/Users/manish/Documents/KU/SEM 5/DL/M

In [10]:
from dotenv import load_dotenv

load_dotenv()

aai.settings.api_key = os.getenv("ASSEMBLYAI_API_KEY")

if not aai.settings.api_key:
    raise ValueError(
        "ASSEMBLYAI_API_KEY not found. Add it to your .env file."
    )

print("AssemblyAI API key loaded from .env")

AssemblyAI API key loaded from .env


In [11]:
from pathlib import Path

PROCESSED_DIR = Path("dataset/processed")

audio_files = [
    file for file in PROCESSED_DIR.rglob("*")
    if file.is_file()
]

print("Number of files:", len(audio_files))

for file in audio_files[:10]:
    print(file)

Number of files: 100
dataset/processed/test/audio_018.wav
dataset/processed/test/audio_019.wav
dataset/processed/test/audio_009.wav
dataset/processed/test/audio_008.wav
dataset/processed/test/audio_011.wav
dataset/processed/test/audio_005.wav
dataset/processed/test/audio_004.wav
dataset/processed/test/audio_010.wav
dataset/processed/test/audio_006.wav
dataset/processed/test/audio_012.wav


In [12]:
test_audio = PROCESSED_DIR / "test" / "audio_000.wav"

print("Audio file:", test_audio)
print("Exists:", test_audio.exists())

if test_audio.exists():
    print("Size:", round(test_audio.stat().st_size / (1024 * 1024), 2), "MB")

Audio file: dataset/processed/test/audio_000.wav
Exists: True
Size: 0.06 MB


In [13]:
config = aai.TranscriptionConfig(
    speech_models=["universal-3-5-pro", "universal-2"],
    speaker_labels=True
)

print("Configuration ready.")

Configuration ready.


In [14]:
transcriber = aai.Transcriber(config=config)

print("Transcriber created.")

Transcriber created.


In [15]:
transcript = transcriber.transcribe(str(test_audio))

print("Transcription finished.")
print("Status:", transcript.status)

Transcription finished.
Status: TranscriptStatus.completed


In [16]:
if transcript.status == aai.TranscriptStatus.error:
    print("ERROR:")
    print(transcript.error)
else:
    print("SUCCESS")

SUCCESS


In [17]:
print(transcript.text)

For example, for the dialogue acts and so on.


In [18]:
if transcript.utterances:
    for utterance in transcript.utterances:
        print(f"Speaker {utterance.speaker}: {utterance.text}")
else:
    print("No speaker utterances found.")

Speaker A: For example, for the dialogue acts and so on.


In [19]:
if transcript.utterances:
    for utterance in transcript.utterances:
        start = utterance.start / 1000
        end = utterance.end / 1000
        
        print(
            f"[{start:.2f}s - {end:.2f}s] "
            f"Speaker {utterance.speaker}: {utterance.text}"
        )

[0.00s - 1.83s] Speaker A: For example, for the dialogue acts and so on.


In [20]:
utterances = []

for utterance in transcript.utterances:
    utterances.append({
        "speaker": utterance.speaker,
        "start": utterance.start / 1000,
        "end": utterance.end / 1000,
        "text": utterance.text
    })

utterances[:5]

[{'speaker': 'A',
  'start': 0.0,
  'end': 1.833,
  'text': 'For example, for the dialogue acts and so on.'}]

In [21]:
df = pd.DataFrame(utterances)

df.head()

,speaker,start,end,text
0,A,0.0,1.833,"For example, for the dialogue acts and so on."


In [22]:
from pathlib import Path

# All AssemblyAI outputs are saved here
OUTPUT_DIR = BASE_DIR / "outputs" / "assemblyai"
JSON_DIR = OUTPUT_DIR / "json"
TRANSCRIPT_DIR = OUTPUT_DIR / "transcripts"

JSON_DIR.mkdir(parents=True, exist_ok=True)
TRANSCRIPT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directories created:")
print("AssemblyAI:", OUTPUT_DIR)
print("JSON:", JSON_DIR)
print("Transcripts:", TRANSCRIPT_DIR)

Output directories created:
AssemblyAI: /Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai
JSON: /Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/json
Transcripts: /Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/transcripts


In [23]:
audio_files = sorted(PROCESSED_DIR.rglob("*.wav"))

print("Total WAV files:", len(audio_files))

for file in audio_files[:10]:
    print(file)

Total WAV files: 100
dataset/processed/test/audio_000.wav
dataset/processed/test/audio_001.wav
dataset/processed/test/audio_002.wav
dataset/processed/test/audio_003.wav
dataset/processed/test/audio_004.wav
dataset/processed/test/audio_005.wav
dataset/processed/test/audio_006.wav
dataset/processed/test/audio_007.wav
dataset/processed/test/audio_008.wav
dataset/processed/test/audio_009.wav


In [24]:
def transcribe_audio(audio_path):
    try:
        transcriber = aai.Transcriber(config=config)

        transcript = transcriber.transcribe(str(audio_path))

        if transcript.status == aai.TranscriptStatus.error:
            return {
                "status": "error",
                "error": transcript.error,
                "audio_file": str(audio_path)
            }

        utterances = []

        if transcript.utterances:
            for utterance in transcript.utterances:
                utterances.append({
                    "speaker": utterance.speaker,
                    "start": utterance.start / 1000,
                    "end": utterance.end / 1000,
                    "text": utterance.text
                })

        return {
            "status": "success",
            "audio_file": str(audio_path),
            "transcript": transcript.text,
            "utterances": utterances
        }

    except Exception as e:
        return {
            "status": "error",
            "audio_file": str(audio_path),
            "error": str(e)
        }

In [25]:
result = transcribe_audio(audio_files[0])

print("Status:", result["status"])

if result["status"] == "success":
    print("\nTranscript:")
    print(result["transcript"])

    print("\nSpeaker utterances:")
    for utterance in result["utterances"]:
        print(
            f"Speaker {utterance['speaker']} "
            f"({utterance['start']:.2f}s - {utterance['end']:.2f}s): "
            f"{utterance['text']}"
        )
else:
    print("Error:", result["error"])

Status: success

Transcript:
For example, for the dialogue acts and so on.

Speaker utterances:
Speaker A (0.00s - 1.83s): For example, for the dialogue acts and so on.


In [26]:
results = []

total_files = len(audio_files)

for i, audio_file in enumerate(audio_files, start=1):
    print(f"\n[{i}/{total_files}] Processing: {audio_file.name}")

    result = transcribe_audio(audio_file)

    results.append(result)

    if result["status"] == "success":
        print("✓ Success")
    else:
        print("✗ Failed:", result["error"])


[1/100] Processing: audio_000.wav
✓ Success

[2/100] Processing: audio_001.wav
✓ Success

[3/100] Processing: audio_002.wav
✓ Success

[4/100] Processing: audio_003.wav
✓ Success

[5/100] Processing: audio_004.wav
✓ Success

[6/100] Processing: audio_005.wav
✓ Success

[7/100] Processing: audio_006.wav
✓ Success

[8/100] Processing: audio_007.wav
✓ Success

[9/100] Processing: audio_008.wav
✓ Success

[10/100] Processing: audio_009.wav
✓ Success

[11/100] Processing: audio_010.wav
✓ Success

[12/100] Processing: audio_011.wav
✓ Success

[13/100] Processing: audio_012.wav
✓ Success

[14/100] Processing: audio_013.wav
✓ Success

[15/100] Processing: audio_014.wav
✓ Success

[16/100] Processing: audio_015.wav
✓ Success

[17/100] Processing: audio_016.wav
✓ Success

[18/100] Processing: audio_017.wav
✓ Success

[19/100] Processing: audio_018.wav
✓ Success

[20/100] Processing: audio_019.wav
✓ Success

[21/100] Processing: audio_000.wav
✓ Success

[22/100] Processing: audio_001.wav
✓ Succe

In [27]:
successful = [
    r for r in results
    if r["status"] == "success"
]

failed = [
    r for r in results
    if r["status"] == "error"
]

print("Total files :", len(results))
print("Successful  :", len(successful))
print("Failed      :", len(failed))

Total files : 100
Successful  : 99
Failed      : 1


In [28]:
for result in failed:
    print(result["audio_file"])
    print("Error:", result["error"])
    print()

dataset/processed/train/audio_027.wav
Error: Audio duration is too short.



In [29]:
for result in successful:
    audio_path = Path(result["audio_file"])

    # Save complete result as JSON
    output_file = JSON_DIR / f"{audio_path.stem}.json"

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(
            result,
            f,
            indent=4,
            ensure_ascii=False
        )

    # Save plain transcript as TXT
    transcript_file = TRANSCRIPT_DIR / f"{audio_path.stem}.txt"

    with open(transcript_file, "w", encoding="utf-8") as f:
        f.write(result["transcript"])

print(f"Saved {len(successful)} JSON files to: {JSON_DIR}")
print(f"Saved {len(successful)} transcript TXT files to: {TRANSCRIPT_DIR}")

Saved 99 JSON files to: /Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/json
Saved 99 transcript TXT files to: /Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/transcripts


In [30]:
rows = []

for result in successful:
    audio_path = Path(result["audio_file"])

    rows.append({
        "audio_file": audio_path.name,
        "transcript": result["transcript"],
        "num_utterances": len(result["utterances"])
    })

assemblyai_df = pd.DataFrame(rows)

assemblyai_csv = OUTPUT_DIR / "assemblyai_transcripts.csv"

assemblyai_df.to_csv(
    assemblyai_csv,
    index=False,
    encoding="utf-8"
)

print("Saved:", assemblyai_csv)

Saved: /Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/assemblyai_transcripts.csv


In [31]:
json_files = sorted(JSON_DIR.glob("*.json"))

print("JSON files:", len(json_files))

for file in json_files[:10]:
    print(file)

JSON files: 79
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/json/audio_000.json
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/json/audio_001.json
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/json/audio_002.json
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/json/audio_003.json
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/json/audio_004.json
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/json/audio_005.json
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/json/audio_006.json
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/json/audio_007.json
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/json/audio_008.json
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/json/audio_009.json


In [32]:
json_files = sorted(JSON_DIR.glob("*.json"))

print("JSON files:", len(json_files))

for file in json_files[:10]:
    print(file)

JSON files: 79
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/json/audio_000.json
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/json/audio_001.json
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/json/audio_002.json
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/json/audio_003.json
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/json/audio_004.json
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/json/audio_005.json
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/json/audio_006.json
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/json/audio_007.json
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/json/audio_008.json
/Users/manish/Documents/KU/SEM 5/DL/Mini Project/outputs/assemblyai/json/audio_009.json


In [33]:
sample_json = JSON_DIR / "audio_000.json"

if sample_json.exists():
    with open(sample_json, "r", encoding="utf-8") as f:
        sample_result = json.load(f)

    print("Audio:", sample_result["audio_file"])
    print("\nTranscript:")
    print(sample_result["transcript"])

    print("\nNumber of utterances:", len(sample_result["utterances"]))

    for utterance in sample_result["utterances"]:
        print(
            f"[{utterance['start']:.2f}s - {utterance['end']:.2f}s] "
            f"Speaker {utterance['speaker']}: "
            f"{utterance['text']}"
        )
else:
    print("Sample JSON not found:", sample_json)

Audio: dataset/processed/train/audio_000.wav

Transcript:
If you SSH in, they have this big warning about doing nothing at all in the gateway machine.

Number of utterances: 1
[0.03s - 4.13s] Speaker A: If you SSH in, they have this big warning about doing nothing at all in the gateway machine.
